# 06 MCP Protocol Design

## MCP 到底在组织什么，以及它为什么不是“再来一层封装”

上一章已经把问题压到了一个比较明确的位置：一旦 Agent 真的要持续调度外部能力，系统就会从“会不会调用函数”迅速升级成“能力如何组织、暴露、发现和治理”。

到了这一步，MCP 的讨论就不能再停留在口号层。它真正值得看的地方，不是它有没有把某些 API 字段标准化，而是它到底选择了什么作为能力系统的基本构件，以及这些构件怎样组成一个可被 Agent Runtime 消费的能力面。

这一章的任务，就是把 MCP 从“听起来合理的协议名词”拆成一套结构判断。重点不在于背术语，而在于看清它的设计重心：**它并不试图替模型思考，也不替 Agent 做任务规划，它做的是把外部世界组织成模型和 runtime 都更容易接入的一层标准化语义接口。**

## 先给结论

如果把 MCP 的协议设计压缩成一句话，可以这样概括：

> MCP 不是围绕“某个函数怎么调”来组织系统，而是围绕“有哪些能力、这些能力属于什么类型、谁来暴露、谁来消费、怎么发现、怎么返回”来组织系统。

这句话背后有几个很关键的设计取向：

- 它把外部能力拆成不同语义类型，而不是全塞进函数调用
- 它承认模型系统不只需要执行动作，也需要读取上下文和复用任务模板
- 它把能力提供方和能力消费方区分开，而不是让每个宿主各写一套私有接法
- 它试图让“能力是什么”先于“这个应用怎么用它”被定义出来

也正因为有这些设计取向，MCP 讨论的重点就不该只是 protocol field，而应该是 capability model。

## 1. MCP 最重要的不是 transport，而是 capability model

很多人第一次看协议时，会把注意力放在通信层和接口形式上。这当然重要，但如果只盯着 transport，很容易错过 MCP 真正有价值的地方。

对 Agent 系统来说，真正难的不是“字节怎么传”，而是“外部能力如何被抽象”。如果抽象不对，后面再优雅的传输层也只是把混乱搬运得更稳定。

MCP 的关键判断在于：外部世界对模型系统来说，并不是单一的一类东西。至少可以明显区分出几种不同语义：

- 有些东西是要执行的，比如调用服务、跑查询、触发操作
- 有些东西是要读取的，比如文档、配置、知识片段、上下文材料
- 有些东西是要作为任务入口复用的，比如 prompt 模板和标准任务骨架

如果把这三类东西都粗暴塞进“函数”里，程序当然还是能跑，但系统会丢掉很重要的语义层次。MCP 的设计价值，就体现在它没有这么做。

## 2. Tools、Resources、Prompts：MCP 的三种一等能力

MCP 里最值得重视的，不是某一个接口，而是它把能力面拆成了三个一等对象：`tools`、`resources`、`prompts`。

这三个对象不是 UI 分类，也不是为了好看才分出来的标签。它们分别对应 Agent 在运行时真正会做的三类事：

- 需要做事时，调用 tool
- 需要知道更多时，读取 resource
- 需要进入某个任务框架时，使用 prompt

这个划分的意义非常大。因为它等于承认：模型系统的外部依赖不只是 action surface，还有 context surface 和 task-entry surface。

如果没有这个划分，所有能力最终都会退化成某种“调用一下函数然后自己解释”的私有写法，语义会越来越混。MCP 通过这三个一等对象，明确给能力层做了结构分区。

## 3. Tool：执行面，不只是函数包装

在 MCP 里，tool 当然和 function calling 有亲缘关系，但不能直接把它缩减成“函数定义的另一种写法”。

从 Agent Runtime 视角看，tool 承担的是执行面职责。它回答的是：当系统已经决定要做一个动作时，这个动作如何被描述、如何被调用、如何返回结果。

这里至少有几个点值得强调：

- tool 不是自由文本，而是带结构和语义边界的能力对象
- tool 的说明不仅给程序看，也给模型和 runtime 看
- tool 的价值不只是执行动作，还在于被纳入可发现、可枚举、可治理的能力面

所以在一个成熟系统里，好的 tool 定义不是“这个函数能跑”，而是“这个动作被描述得足够清楚，以至于 Agent 能稳定地判断何时使用它，以及如何消费它的结果”。

## 4. Resource：MCP 最容易被低估、也最关键的对象之一

如果只把 MCP 看成工具协议，就会错过它最有洞见的部分之一：resource 被提升成第一等能力。

这个设计很重要，因为很多实际任务根本不是先做动作，而是先读材料。一个 Agent 在执行过程中，常常需要：

- 读项目背景
- 读一段产品文档
- 读某个接口说明
- 读一份团队规范
- 读一个本地文件或知识节点

这些行为不适合被粗暴建模成 tool call，因为它们的核心不是动作副作用，而是上下文摄取。把它们独立成 resource，等于是在协议层承认“上下文读取”本身就是一种被治理的能力。

这会带来两个直接收益：

- Agent 可以明确区分“我是在做事”还是“我是在读资料”
- 能力提供方可以用更稳定的方式暴露只读上下文，而不必把所有东西都伪装成工具输出

从系统设计角度看，这是 MCP 比普通工具注册更像能力协议的原因之一。

把 resource 想得更具体一点，会更容易理解它为什么值得成为第一等对象：

- `repo://architecture/overview`：项目架构总览，用于回答系统设计相关问题
- `product://requirements/current-prd`：当前版本产品需求文档

这类对象的价值不在于“能读文件”，而在于它们被稳定组织成了任务可读取的上下文入口。

## 5. Prompt：为什么任务模板也应该协议化

很多人对 prompt 被纳入能力层会有一点直觉阻力，觉得 prompt 只是调用方自己的事情，为什么还要进入协议。

如果系统只是一个很小的单体应用，这种想法问题不大。但一旦你真的要构建可复用的 Agent 生态，就会发现任务入口本身也值得被治理。

原因很简单：很多高价值任务并不是从一条任意用户输入就能稳定开始，而是需要一个经过设计的入口框架，比如：

- 如何分析一份 PRD
- 如何把 JD 映射成候选人能力维度
- 如何把一段代码仓库说明转成集成方案

这些都不是纯知识，也不是执行动作，而更像“结构化任务入口”。如果这种入口只散落在某个应用内部，它就无法被其他宿主、其他 Agent、其他工作流稳定复用。

把 prompt 也纳入协议层，本质上是在说：任务模板本身也是能力。它不是最终答案，但它会决定系统进入问题的方式。

## 6. Host、Client、Server：MCP 的角色拆分为什么重要

除了能力对象本身，MCP 的另一个关键设计，是它明确区分了能力提供方和能力消费方。这个区分听起来平常，但对系统清晰度非常关键。

粗略地说，可以这样理解这几个角色：

- `server`：暴露能力，告诉外部我有哪些 tools/resources/prompts 可供使用
- `client`：连接和调用这些能力，把协议层操作转成实际使用
- `host`：承载模型与 runtime 的应用环境，决定这些能力如何被纳入具体任务流程

把这些角色拆开，有两个直接好处：

- 能力定义不再被硬编码在单一宿主里
- 消费能力的系统可以变化，但能力提供方式保持一致

也就是说，MCP 不是在写某个应用的插件接口，而是在定义一套更中立的能力交换边界。

这几个角色直接写开就够了：

- `host`：承载模型和 Agent Runtime 的应用环境
- `client`：负责发现、请求和消费 MCP 能力
- `server`：负责暴露 tools、resources、prompts 的能力提供方

角色拆分的重点不是术语，而是让“谁提供能力、谁消费能力、谁承载任务流程”这三件事不再混在一起。

## 7. MCP 同时有语义层和执行层，但重心在语义层

一个容易被忽略的点是：MCP 当然会落到请求和返回这种执行接口上，但它的核心贡献并不只是“定义了一组能调的方法”。它更重要的地方在于，它先定义了外部能力在语义上是什么。

这点特别重要，因为模型系统不是普通 RPC 客户端。它需要的不是纯粹的函数可达性，而是：

- 这些能力各自属于什么类型
- 这些能力大概适合什么场景
- 这些能力返回的结果应该被当成什么来处理

如果没有这个语义层，执行层再完整，最终也很容易退化成“能调，但很难稳定用好”。而 MCP 的设计正是试图让能力既可执行，又可理解。

## 8. Discoverability：为什么“能被看见”本身就是协议价值

传统应用里，很多能力其实是存在的，只是存在于私有代码和局部常识里。开发者知道它们在哪，但系统生态并不知道。MCP 把 discoverability 拉到前台，本质上是在把“局部常识”变成“协议级目录”。

这件事的重要性，在单体 demo 里不明显，但在多宿主、多 Agent、多团队协作时会被放大。因为真正可扩展的能力系统，不应该要求每个消费方都先去翻一遍源代码，才能知道外面能做什么、能读什么、能从哪种任务模板开始。

可发现性不是文档友好度问题，而是生态可扩展性问题。MCP 让能力先可见，再可用，这个顺序很关键。

## 9. Uniform Capability Surface：统一能力面比单个工具更重要

如果从产品演示视角看，一个很酷的工具往往最吸引注意力；但从系统设计视角看，真正重要的通常不是某个工具有多厉害，而是所有能力能不能形成一个统一能力面。

统一能力面意味着：

- 能力对象的描述风格相对一致
- 调用和读取方式有共同结构
- 不同宿主看到的能力元信息尽量一致
- Agent Runtime 能在一个稳定接口模型上做决策，而不是每接一个能力源就重写一层胶水

这正是 MCP 的真正价值区间。它不是把单个能力做强，而是把能力生态做平。对 Agent 来说，后者往往更重要。

## 10. MCP 为什么不是 Agent 框架，但又天然适合 Agent

这里必须再强调一次边界：MCP 不是 Agent 框架。它不负责目标管理，不负责状态推进，不负责多步规划，也不负责终止条件。

但它又天然适合 Agent，原因也很明确：Agent 本质上需要一个被组织良好的外部能力层，而 MCP 正是在提供这层东西。

更具体地说：

- Agent 决定什么时候该读 resource
- Agent 决定什么时候该调 tool
- Agent 决定什么时候该套用某个 prompt 模板
- MCP 负责把这些对象稳定地摆在那儿，供 Agent 以统一方式消费

所以，MCP 和 Agent 的关系最合理的理解不是“二选一”，而是“任务层依赖能力层”。前者没有后者会越来越乱，后者没有前者则不会自己完成任务。

## 11. MCP 的设计取舍：不是一切都更自动，而是一切都更可组织

协议化从来不是免费的。只要引入一层标准接口，就会带来额外的抽象、额外的定义工作、额外的能力建模成本。MCP 也不例外。

所以它的价值不应该被理解成“用了以后所有问题都自动解决”，而应该理解成另一种取舍：你付出一些抽象成本，换取能力组织的长期清晰度。

这种取舍只有在一个前提下值得做，就是系统真的要超越单体 demo，进入可扩展、可复用、可协作的能力生态。对一个只会用到两个本地函数的一次性脚本，MCP 当然可能显得重；但对一个希望让 Agent 能稳定消费外部能力的系统，它会越来越合理。

## 12. 下一步为什么应该直接做一个本地 MCP Server

到这里，协议层的价值已经足够清楚，下一步最自然的动作就不该继续停留在抽象概念，而是把能力面真的落出来。

做一个本地 MCP Server 的意义，不只是为了 demo 一个 server 能启动，而是为了把前面几章的判断都放到一个可运行对象里验证：

- tool 到底应该怎么描述才像能力对象
- resource 到底应该怎样暴露才像上下文入口
- prompt 到底为什么值得成为独立能力
- Agent Runtime 消费的，应该是怎样一个统一能力面

也就是说，本地 server 不是配套练习，而是这套体系第一次真正落到系统形态上的地方。

## 13. 本章结论

这一章最值得保留的判断有这些：

- MCP 的核心不是 transport，而是 capability model。
- tools、resources、prompts 的三分法，体现的是 Agent 对外部世界的三种真实需求。
- resource 被提升为第一等对象，是 MCP 比普通工具注册更像能力协议的关键原因之一。
- host、client、server 的角色拆分，让能力定义和能力消费不再绑死在单一宿主里。
- MCP 不是 Agent 框架，但它天然适合作为 Agent 的能力层。

下一章会把这些判断直接落进一个本地 MCP Server 的设计里，不再只讨论概念，而是讨论怎样把本地能力真正组织成一个有结构、可被 Agent 消费、也能被后续 notebook 继续复用的能力面。